# Text-to-speech and speech recognition

In [1]:
import os
import jiwer
import glob
import scipy
import torch
import librosa
import parselmouth
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from IPython.display import Audio
from parselmouth.praat import call
from transformers import AutoTokenizer, VitsModel, WhisperProcessor, WhisperForConditionalGeneration

/data/mtseng/miniconda3/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
sentences_df = pd.read_csv('./hw2_data/sentences.csv')

## Problem 3(a): Text to speech

In [3]:
tok = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")
tts_model = VitsModel.from_pretrained("facebook/mms-tts-eng")

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

In [4]:
synthesized_output_dir = './output/problem3/synthesized'
os.makedirs(synthesized_output_dir, exist_ok=True)

torch.manual_seed(42)
with torch.no_grad():
    for _, row in sentences_df.iterrows():
        output_path = os.path.join(synthesized_output_dir, f"sentence_{row['id']}.wav")
        if os.path.exists(output_path):
            continue
        text = row['text']
        ids = tok(text, return_tensors="pt")
        synthesized_audio = tts_model(**ids).waveform[0]
        scipy.io.wavfile.write(output_path, tts_model.config.sampling_rate, synthesized_audio.detach().numpy())

## Problem 3(b): Speech recognition

Transcribe the 12 synthetic sentences and the six recordings in
speech_clips/ with both Whisper models, using greedy decoding and telling the model that the
language is English, and compute the WER of every transcript against its reference. Report the
mean WER by sentence type for the synthetic speech, and a summary table with one row per
model and columns for synthetic speech, real clean speech and real noisy speech. Quote the
synthetic sentence with the highest WER for whisper-base both as it was said and as it was heard.

In [5]:
transcript_path = "./output/problem3/transcripts.csv"
if not os.path.exists(transcript_path):
    os.makedirs(os.path.dirname(transcript_path), exist_ok=True)
    model_types = ["openai/whisper-tiny", "openai/whisper-base"]
    transcripts_df = []
    for model_type in model_types:
        processor = WhisperProcessor.from_pretrained(model_type)
        asr_model = WhisperForConditionalGeneration.from_pretrained(model_type)
        asr_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task="transcribe")

        sample_audios = glob.glob("./hw2_data/speech_clips/*.wav") + glob.glob(f'{synthesized_output_dir}/*.wav')

        batch_audio_data = []
        for audio_path in sample_audios:
            audio_data, _ = librosa.load(audio_path, sr=16000)
            batch_audio_data.append(audio_data)
        input_features = processor(batch_audio_data, sampling_rate=16000, return_tensors="pt").input_features 
        predicted_ids = asr_model.generate(input_features)
        transcripts = processor.batch_decode(predicted_ids, skip_special_tokens=True)
        for audio_path, transcript in zip(sample_audios, transcripts):
            transcripts_df.append({
                "model_type": model_type,
                "file": os.path.basename(audio_path),
                "transcript": transcript
            })
    transcripts_df = pd.DataFrame(transcripts_df)
    transcripts_df.to_csv("./output/problem3/transcripts.csv", index=False)
else:
    print("Loading existing transcripts from", transcript_path)
    transcripts_df = pd.read_csv(transcript_path)

print(transcripts_df.head())

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may obser

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

            model_type                file  \
0  openai/whisper-tiny  my_voice_noisy.wav   
1  openai/whisper-tiny        noisy_lj.wav   
2  openai/whisper-tiny       noisy_jfk.wav   
3  openai/whisper-tiny     noisy_libri.wav   
4  openai/whisper-tiny        clean_lj.wav   

                                          transcript  
0   When the sunlight strikes ring drop in the ai...  
1   For although the Chinese took impressions fro...  
2   And so my fellow Americans, and not what your...  
3   Experience proves this. Why should one fall o...  
4   For although the Chinese took impressions fro...  


In [6]:
gt_transcripts_df_1 = pd.read_csv("./hw2_data/speech_clips/references.csv")
gt_transcripts_df_2 = pd.read_csv("./hw2_data/sentences.csv")
with open("./hw2_data/my_script.txt", "r") as f:
    my_voice_transcript = f.read()

gt_transcripts_df = []
for _, row in gt_transcripts_df_1.iterrows():
    gt_transcripts_df.append({
        "file": row["file"],
        "transcript": row["reference"],
        "sentence_type": ""
    })
for _, row in gt_transcripts_df_2.iterrows():
    gt_transcripts_df.append({
        "file": "sentence_{}.wav".format(row["id"]),
        "transcript": row["spoken"],
        "sentence_type": row["type"]
    })
for my_voice_type in ["my_voice", "my_voice_noisy"]:
    gt_transcripts_df.append({
        "file": "{}.wav".format(my_voice_type),
        "transcript": my_voice_transcript,
        "sentence_type": ""
    })
gt_transcripts_df = pd.DataFrame(gt_transcripts_df)

In [7]:
wer_results = []
for _, row in transcripts_df.iterrows():
    gt_row = gt_transcripts_df[gt_transcripts_df["file"] == row["file"]]
    if gt_row.empty:
        raise ValueError("No ground truth found for file: {}".format(row["file"]))

    gt_transcript = gt_row.iloc[0]["transcript"]
    wer_results.append({
        'model': row['model_type'],
        'file': row['file'],
        'wer': jiwer.wer(gt_transcript, row["transcript"]),
        'audio_type': 'synthesized' if 'sentence' in row['file'] else ("noisy" if 'noisy' in row['file'] else "clean"),
        'sentence_type': gt_row.iloc[0]["sentence_type"],
        'predicted_transcript': row["transcript"],
        'gt_transcript': gt_transcript
    })
wer_results = pd.DataFrame(wer_results)


In [8]:
selected_wer_results = wer_results[~wer_results['file'].str.contains('my_voice')].reset_index(drop=True)

table = []
for model_type in selected_wer_results['model'].unique():
    audio_type_means = {}
    for audio_type in selected_wer_results['audio_type'].unique():
        subset = selected_wer_results[(selected_wer_results['model'] == model_type) & (selected_wer_results['audio_type'] == audio_type)]
        mean_wer = subset['wer'].mean()
        audio_type_means[audio_type] = mean_wer

    table.append({
        'model': model_type,
        **audio_type_means
    })

# Report the mean WER by sentence type for the synthetic speech
sentence_type_table = []
for sentence_type in selected_wer_results['sentence_type'].unique():
    if sentence_type == "":
        continue
    subset = selected_wer_results[(selected_wer_results['audio_type'] == 'synthesized') & (selected_wer_results['sentence_type'] == sentence_type)]
    mean_wer = subset['wer'].mean()
    sentence_type_table.append({
        'sentence_type': sentence_type,
        'mean WER': mean_wer
    })
print("Mean WER by sentence type for synthesized speech:")
sentence_type_table = pd.DataFrame(sentence_type_table).sort_values(by='mean WER', ascending=False).reset_index(drop=True)
display(sentence_type_table)

print("Mean WER by audio type for each model:")
table = pd.DataFrame(table)
display(table)

# find the model with the highest WER for whisper-base
sorted_wer_results = selected_wer_results[(selected_wer_results['model'] == 'openai/whisper-base') & (selected_wer_results['audio_type'] == 'synthesized') ].sort_values(by='wer', ascending=False)
highest_wer_row = sorted_wer_results.iloc[0]

print("Sentence with highest WER for whisper-base (synthesized):")
print("File:", highest_wer_row['file'])
print("WER:", highest_wer_row['wer'])
print("What was said:", highest_wer_row['gt_transcript'])
print("What was heard:", highest_wer_row['predicted_transcript'])


Mean WER by sentence type for synthesized speech:


,sentence_type,mean WER
0,name,0.857143
1,abbreviation,0.743056
2,numbers,0.730556
3,plain,0.481481
4,homograph,0.250000
5,long,0.204545


Mean WER by audio type for each model:


,model,noisy,clean,synthesized
0,openai/whisper-tiny,0.297089,0.277481,0.578051
1,openai/whisper-base,0.312240,0.266786,0.535934


Sentence with highest WER for whisper-base (synthesized):
File: sentence_2.wav
WER: 1.0
What was said: she sells seashells by the seashore
What was heard:  She sells sea shells but the sea shore.


Discuss which sentence types are hardest and what kind of “error” many of them turn out to be,
how whisper-tiny compares with whisper-base, and how recognition of synthetic speech
compares with recognition of real clean and real noisy speech.

In [9]:
# TODO

## Problem 3(c): Does ASR survive PSOLA?

In [ ]:
# TODO

## Problem 3(d): Your own voice

Finally, test the recognizers on yourself. Transcribe my_voice and my_voice_noisy with both
Whisper models, and also synthesize the text of my_script.txt with the TTS model and transcribe
that. Report the WER against the script for all three kinds of speech and both models, and show
the whisper-base transcript of each. Discuss which words were recognized wrongly and why, how
much the noise hurt, and whether it was easier for Whisper to recognize you or the synthetic voice.

In [10]:
tts_audio_path = './output/problem3/my_script_tts.wav'
if not os.path.exists(tts_audio_path):
    ids = tok(my_voice_transcript, return_tensors="pt")
    synthesized_audio = tts_model(**ids).waveform[0]
    scipy.io.wavfile.write(tts_audio_path, tts_model.config.sampling_rate, synthesized_audio.detach().numpy())

synthesized_row = []
for model_type in ["openai/whisper-tiny", "openai/whisper-base"]:
    processor = WhisperProcessor.from_pretrained(model_type)
    asr_model = WhisperForConditionalGeneration.from_pretrained(model_type)
    asr_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task="transcribe")
    audio_data, _ = librosa.load(tts_audio_path, sr=16000)
    input_features = processor(audio_data, sampling_rate=16000, return_tensors="pt").input_features 
    predicted_ids = asr_model.generate(input_features)
    transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    synthesized_row.append({
        "model": model_type,
        "file": os.path.basename(tts_audio_path),
        "wer": jiwer.wer(my_voice_transcript, transcript),
        "audio_type": "tts",
        "sentence_type": "",
        "predicted_transcript": transcript,
        "gt_transcript": my_voice_transcript
    })
synthesized_row = pd.DataFrame(synthesized_row)
    

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

In [11]:
selected_wer_results = wer_results[wer_results['file'].str.contains('my_voice')].copy().reset_index(drop=True)
selected_wer_results = pd.concat([selected_wer_results, synthesized_row], ignore_index=True)

In [12]:
table = []
for model_type in selected_wer_results['model'].unique():
    table.append({
        "model": model_type,
        "clean": selected_wer_results[(selected_wer_results['model'] == model_type) & (selected_wer_results['audio_type'] == 'clean')]['wer'].mean(),
        "noisy": selected_wer_results[(selected_wer_results['model'] == model_type) & (selected_wer_results['audio_type'] == 'noisy')]['wer'].mean(),
        "tts": selected_wer_results[(selected_wer_results['model'] == model_type) & (selected_wer_results['audio_type'] == 'tts')]['wer'].mean()
    })
table = pd.DataFrame(table)
display(table)

,model,clean,noisy,tts
0,openai/whisper-tiny,0.235294,0.137255,0.235294
1,openai/whisper-base,0.196078,0.098039,0.215686


In [17]:
selected_wer_results_base = selected_wer_results[selected_wer_results['model'] == 'openai/whisper-base']
print("GT: ", my_voice_transcript)
for _, row in selected_wer_results_base.iterrows():
    print(row['file'])
    print(row['predicted_transcript'])
    print("=="*20)

GT:  When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round arch, with its path high above, and its two ends apparently beyond the horizon.

my_voice_noisy.wav
 With the sunlight strikes raindrops in the air, they act as a prison and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round art, with its path height above and its two ends apparently beyond the horizon.
my_voice.wav
 When the sunlight strikes rain drop in the air, they access the prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long-round art, with its path high above and its two ends, apparently beyond the horizon.
my_script_tts.wav
 When the sun likes dry rain, drops in the air. They act as a prism and form a rainbow. The rainbow is a division of w